In [ ]:
import numpy as np
import xarray as xr
import somoclu
import pandas as pd
import geopandas as gpd

#add path and data
path1=' '
in_path=path1+" "

gph_anom=xr.open_dataarray(path1+"uyr_gph_anom_1979-2022.nc")  #Detrended + anomalous gph data
print(gph_anom)
time=gph_anom.time.values
lat=gph_anom.latitude.values
lon=gph_anom.longitude.values

shp = gpd.read_file(in_path+"Upper_Yangtze_River.shp") #study area shp file

def clip_arr(arr):
    arr.rio.write_crs("epsg:4326", inplace=True)
    arr.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=True)
    clip_data=arr.rio.clip(shp.geometry, shp.crs, drop=False, all_touched=True)
    return clip_data

clip_da=clip_arr(gph_anom)  

#Data prepared for input into the SOM model, null values removed
data=[]
for t in range(len(time)):
    da1=clip_da[t,:,:].values.flatten()
    arr = da1[~np.isnan(da1)]
    data.append(arr)
data=np.array(data) 

In [ ]:
###SOM model training and result storage###
som_grids = {
    'grid3x3': (3, 3),
    'grid3x4': (3, 4),
    'grid4x4': (4, 4),
}

def train_som(grid_size,data_matrix):
    som = somoclu.Somoclu(n_rows=grid_size[0], n_columns=grid_size[1], maptype='planar', gridtype='rectangular', initialization='pca')
    som.train(data=data_matrix, epochs=100, scale0=0.05, scaleN=0.01)
    return som

for grid_name, grid_size in som_grids.items():
    
    print(f"Processing with {grid_name}")

    # training
    print("start som training")
    som=train_som(grid_size,data)

    weights = som.codebook 
    weights_df = pd.DataFrame(weights.reshape(-1,weights.shape[2])) 
    weights_df.to_csv(in_path+f"gph_{grid_name}_total_weights.csv", index=False)

    bmu = som.bmus
    bmu_df = pd.DataFrame(bmu)
    num_cols = grid_size[1]
    bmu_df['node'] = (bmu_df[1] * num_cols + bmu_df[0])+1
    bmu_df['node'].to_csv(in_path+f"gph_{grid_name}_total_classif.csv", index=False)

In [ ]:
# Calculation of model training metrics
from sklearn.metrics import davies_bouldin_score, silhouette_score

som_grids = {
    'grid2x2': (2, 2),
    'grid2x3': (2, 3),
    'grid3x3': (3, 3),
    'grid3x4': (3, 4),
    'grid4x4': (4, 4),
    'grid4x5': (4, 5),
    'grid5x5': (5, 5),
}

eval_results = []

def train_som(grid_size,data_matrix):
    som = somoclu.Somoclu(n_rows=grid_size[0], n_columns=grid_size[1], maptype='planar', gridtype='rectangular', initialization='pca')
    som.train(data=data_matrix, epochs=100, scale0=0.05, scaleN=0.01)
    return som

for grid_name, grid_size in som_grids.items():   
    print(f"Processing with {grid_name}")

    data_matrix=data.astype(np.float32)

    print("start som training")
    som=train_som(grid_size,data_matrix)

    surface_state = som.get_surface_state(data_matrix)

    best_indices = np.argsort(surface_state, axis=1)
    bmu1_idx = best_indices[:, 0]
    bmu2_idx = best_indices[:, 1]

    num_cols = grid_size[1]
    bmu1_x, bmu1_y = bmu1_idx % num_cols, bmu1_idx // num_cols
    bmu2_x, bmu2_y = bmu2_idx % num_cols, bmu2_idx // num_cols

    #  QE
    matched_w = som.codebook[bmu1_y.astype(int), bmu1_x.astype(int)]
    # 计算欧氏距离的均值
    qe = np.mean(np.linalg.norm(data_matrix - matched_w, axis=1))

    #  TE
    dx = np.abs(bmu1_x - bmu2_x)
    dy = np.abs(bmu1_y - bmu2_y)
    te_errors = np.where((dx > 1) | (dy > 1), 1, 0)
    te = np.mean(te_errors)

    #  DBI
    dbi = davies_bouldin_score(data_matrix, bmu1_idx)

    # Silhouette Score 
    sil_score = silhouette_score(data_matrix, bmu1_idx)
    u_matrix_values = som.umatrix

    #print and storage results
    print(f"QE: {qe:.4f} | TE: {te:.4f}| DBI: {dbi} | Silhouette: {sil_score}")
    eval_results.append({
        'Grid': grid_name, 
        'QE': qe, 
        'TE': te,
        'DBI': dbi,
        'Silhouette': sil_score,
        'U_Mean': np.mean(u_matrix_values)
    })
    
pd.DataFrame(eval_results).to_excel(in_path+f".xlsx", index=False)